In [31]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain.agents import create_agent
from pinecone import Pinecone, ServerlessSpec
from langchain_core.tools import tool
import time

In [32]:
# 获取脚本所在目录
try:
    SCRIPT_DIR = Path(__file__).parent
except NameError:
    SCRIPT_DIR = Path.cwd()
DATA_DIR = SCRIPT_DIR / "data"
# 确保 data目录存在
DATA_DIR.mkdir(exist_ok=True)
# 加载环境变量
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY or GROQ_API_KEY == "your_groq_api_key_here":
    raise ValueError("Please set your GROQ_API_KEY in .env file")
# 初始化模型
model = init_chat_model("groq:llama-3.3-70b-versatile", api_key=GROQ_API_KEY)
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
if not PINECONE_API_KEY or PINECONE_API_KEY == "your_pinecone_api_key_here":
    raise ValueError("Please set your PINECONE_API_KEY in .env file")

In [33]:
def example():
    # 1.文档加载
    print("\n[1/6] 文档加载...")
    sample_text = """
    LangChain 是一个用于构建 LLM 应用的框架。

    它提供了以下核心组件：
    1. Models - 语言模型接口
    2. Prompts - 提示词模板
    3. Chains - 链式调用
    4. Agents - 智能代理

    RAG (Retrieval-Augmented Generation) 是 LangChain 的核心应用场景之一。
    """
    doc_path = DATA_DIR / "langchain_intro.txt"
    with open(doc_path, "w", encoding="utf-8") as f:
        f.write(sample_text)
    loader = TextLoader(doc_path, encoding="utf-8")
    documents = loader.load()
    print(f"[1/6] 文档加载完成，共 {len(documents)} 个文档")

    # 2.文本分割
    print("\n[2/6] 文本分割...")
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=200,   # 分块大小
        chunk_overlap=50, # 分块重叠
        separators=["\n\n", "\n", "。", "！", "？", " ", ""]
    )
    chunks = splitter.split_documents(documents)
    print(f"[2/6] 文档分割完成，共 {len(chunks)} 个分块")

    # 3.向量化测试
    print("\n[3/6] 向量化测试...")
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L12-v2"
    )
    vector = embeddings.embed_query("LangChain 是什么")
    print(f"[3/6] 向量化测试，向量维度为 {len(vector)}")

    # Pinecone
    if PINECONE_API_KEY:
        print("\n[4/6] Pinecone 设置...")
        try:
            pc = Pinecone(api_key=PINECONE_API_KEY)
            index_name = "langchain-rag-demo"
            dimension = 384

            existing_indexes = [idx.name for idx in pc.list_indexes()]
            if index_name not in existing_indexes:
                print(f"[OK] 索引已存在")
                index = pc.Index(index_name)
            else:
                print("创建新索引")
                pc.create_index(
                    name=index_name,
                    dimension=dimension,
                    metric="cosine",        # 余弦相似度
                    spec=ServerlessSpec(
                        cloud="aws",
                        region="us-east-1"
                    )
                )
                time.sleep(10)
                index = pc.Index(index_name)
                print("[OK] 索引创建成功")

            print("[5/6] 文档向量化索引...")
            vectorstore = PineconeVectorStore.from_documents(
                documents=chunks,
                embeddings=embeddings,
                index_name=index_name
            )
            print(f"[OK] {len(chunks)} 个文档块已索引")

            print("\n[6/6] RAG 问答...")
            @tool
            def search_knowledge_base(query: str) -> str:
                """
                使用 Pinecone 索引进行知识库搜索相关信息

                参数
                    query：用户问题

                返回值
                    字符串类型的topk检索回答
                """
                docs = vectorstore.similarity_search(query, k=2)
                return "\n\n".join([doc.page_content for doc in docs])

            agent = create_agent(
                model=model,
                tools=[search_knowledge_base],
                system_prompt="你是一个助手，可以访问知识库。使用 search_knowledge_base 工具检索相关信息，然后回答问题"
            )

            question = "LangChain 有哪些核心组件"
            print(f"\n 问答：{question}")
            try:
                response = agent.invoke({
                    "messages": [
                        {"role": "user", "content": question}
                    ]
                })
                print(f" 回答：{response['messages'][-1].content}")
                print(f"\n [OK] RAG 回答完成")
            except Exception as e:
                print(f"[错误] RAG 错误：{e}")
        except Exception as e:
            print(f"[错误] Pinecone 设置失败：{e}")
    else:
        print("\n[4-6] 未设置 Pinecone API key")

In [34]:
def main():
    print("\n"+"="*70)
    print("RAG Basic")
    print("="*70)

    example()

In [ ]:
if __name__ == "__main__":
    main()